# Notebook 2 — Chunking and Embedding

This notebook takes the raw 3GPP PDF pages from Notebook 1 and prepares them
for storage in ChromaDB. We split each page into overlapping chunks so the
retrieval system can return precise, self-contained passages rather than entire
pages. We then convert each chunk into a vector embedding using OpenAI and
store everything locally in ChromaDB.

In [1]:
import sys
from dotenv import load_dotenv
import os

load_dotenv()

print(sys.executable)  # confirm rag3gpp environment
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))

c:\Users\nabee\anaconda3\envs\rag3gpp\python.exe
OpenAI key loaded: True


## Loading and chunking the documents

We load all three PDFs and split them into overlapping chunks using LangChain's
RecursiveCharacterTextSplitter. The chunk size of 1000 characters balances
context richness against retrieval precision — large enough to capture a full
technical clause, small enough that the LLM isn't flooded with irrelevant text.
The 200 character overlap prevents information loss at chunk boundaries.

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# load all three specs
pdf_files = [
    r"C:\Projects\3gpp-rag\data\raw\TS_38211.pdf",
    r"C:\Projects\3gpp-rag\data\raw\TS_38214.pdf",
    r"C:\Projects\3gpp-rag\data\raw\TS_38300.pdf",
]

all_pages = []
for path in pdf_files:
    loader = PyPDFLoader(path)
    all_pages.extend(loader.load())

print(f"Total pages loaded: {len(all_pages)}")

# split pages into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]  # tries paragraph breaks first, then line breaks
)

chunks = splitter.split_documents(all_pages)

print(f"Total chunks created: {len(chunks)}")
print(f"\nExample chunk:\n{chunks[10].page_content}")
print(f"\nExample chunk metadata: {chunks[10].metadata}")

Total pages loaded: 841
Total chunks created: 3697

Example chunk:
5.2.1 Pseudo-random sequence generation ......................................................................................................... 18 
5.2.2 Low-PAPR sequence generation type 1 ..................................................................................................... 18 
5.2.2.1 Base sequences of length 36 or larger .................................................................................................. 18 
5.2.2.2 Base sequences of length less than 36 .................................................................................................. 19 
5.2.3 Low-PAPR sequence generation type 2 ..................................................................................................... 22 
5.2.3.1 Sequences of length 30 or larger .......................................................................................................... 22

Example chunk metadata: {'producer': 

## Generating embeddings and storing in ChromaDB

We convert each chunk into a 1536-dimensional vector using OpenAI's
text-embedding-3-small model, then store the vectors locally in ChromaDB.
This is a one-time operation — once the database is built we query it
without re-embedding. We use a persistent ChromaDB client so the vectors
survive between sessions.

In [4]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# set up the embedding model
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# this will embed all 3697 chunks and store them to disk
# takes 3-5 minutes and costs roughly $0.05
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=r"C:\Projects\3gpp-rag\chroma_db"
)

print(f"Chunks stored in ChromaDB: {vectorstore._collection.count()}")

Chunks stored in ChromaDB: 3697


## Testing retrieval

A quick sanity check to confirm ChromaDB can take a natural language query,
find the most semantically similar chunks, and return them with their source
metadata. This is the core operation the RAG chain will rely on.

In [5]:
# load the existing vectorstore from disk (no re-embedding needed)
vectorstore = Chroma(
    persist_directory=r"C:\Projects\3gpp-rag\chroma_db",
    embedding_function=embeddings
)

# test query relevant to the specs we loaded
query = "How is the physical downlink shared channel modulated in NR?"

results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata.get('title')} | Page: {doc.metadata.get('page')}")
    print(f"Content: {doc.page_content[:300]}")
    print()

--- Result 1 ---
Source: 3GPP TS 38.214 | Page: 36
Content: used in the physical downlink shared channel. 
elseif the UE is configured with the higher layer parameter mcs-Table given by SPS-Config set to 'qam64LowSE' 
- if the PDSCH is scheduled by a PDCCH with CRC scrambled by CS -RNTI or 
- if the PDSCH is scheduled without corresponding PDCCH transmission

--- Result 2 ---
Source: 3GPP TS 38.300 | Page: 56
Content: RAN; 
- support for HARQ; 
- support for dynamic link adaptation by varying the transmit power, modulation and coding ; 
- support for SL discontinuous reception (SL DRX) to enable UE power saving.  
Association of transport channels to physical channels is described in TS 38.202 [ 20]. 
5.6 Access 

--- Result 3 ---
Source: 3GPP TS 38.211 | Page: 4
Content: 7.2 Physical resources .......................................................................................................................................... 118 
7.3 Physical channels ............................

## Rebuilding ChromaDB with full 8-spec corpus

Re-running ingestion with all 8 specs loaded. We delete the existing ChromaDB
directory first to avoid duplicate chunks from the original 3-spec build.

In [1]:
import shutil
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# delete the existing ChromaDB so we start fresh
chroma_path = r"C:\Projects\3gpp-rag\chroma_db"
if os.path.exists(chroma_path):
    shutil.rmtree(chroma_path)
    print("Existing ChromaDB deleted")

# load all 8 specs
raw_dir = r"C:\Projects\3gpp-rag\data\raw"
pdf_files = sorted([f for f in os.listdir(raw_dir) if f.endswith(".pdf")])

all_pages = []
for filename in pdf_files:
    path = os.path.join(raw_dir, filename)
    loader = PyPDFLoader(path)
    pages = loader.load()
    all_pages.extend(pages)
    print(f"Loaded {filename}: {len(pages)} pages")

print(f"\nTotal pages: {len(all_pages)}")

# chunk everything
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)
chunks = splitter.split_documents(all_pages)
print(f"Total chunks: {len(chunks)}")

# embed and store - this will take 15-20 minutes
print("\nStarting embedding... do not interrupt")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=chroma_path
)

print(f"\nDone. Chunks stored in ChromaDB: {vectorstore._collection.count()}")

Loaded TS_38211.pdf: 171 pages
Loaded TS_38212.pdf: 327 pages
Loaded TS_38213.pdf: 331 pages
Loaded TS_38214.pdf: 356 pages
Loaded TS_38215.pdf: 38 pages
Loaded TS_38300.pdf: 314 pages
Loaded TS_38331.pdf: 1902 pages
Loaded TS_38401.pdf: 184 pages

Total pages: 3623
Total chunks: 15065

Starting embedding... do not interrupt

Done. Chunks stored in ChromaDB: 15065


In [2]:
# quick retrieval test on the expanded corpus
vectorstore = Chroma(
    persist_directory=r"C:\Projects\3gpp-rag\chroma_db",
    embedding_function=embeddings
)

test_queries = [
    "What is the subcarrier spacing for NR numerology?",
    "How does beamforming work in massive MIMO?",
    "What are the HARQ retransmission procedures?"
]

for query in test_queries:
    docs = vectorstore.similarity_search(query, k=2)
    print(f"Query: {query}")
    for doc in docs:
        print(f"  → {doc.metadata.get('title')} | Page {doc.metadata.get('page')}")
    print()

Query: What is the subcarrier spacing for NR numerology?
  → 3GPP TS 38.331 | Page 815
  → 3GPP TS 38.331 | Page 539

Query: How does beamforming work in massive MIMO?
  → 3GPP TS 38.300 | Page 284
  → 3GPP TS 38.300 | Page 116

Query: What are the HARQ retransmission procedures?
  → 3GPP TS 38.300 | Page 50
  → 3GPP TS 38.300 | Page 136



## Rebuilding ChromaDB with full 14-spec corpus

Added 6 additional specs including TR 38.843 (AI/ML for NR air interface)
and TR 38.873 (MIMO enhancements) to bring the corpus to 14 specifications
covering the complete NR physical layer stack, architecture, and 6G/AI study
items. Total corpus: 4,493 pages.

In [1]:
import os
import time
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
import chromadb

chroma_path = r"C:\Projects\3gpp-rag\chroma_db"

raw_dir = r"C:\Projects\3gpp-rag\data\raw"
pdf_files = sorted([f for f in os.listdir(raw_dir) if f.endswith(".pdf")])

all_pages = []
for filename in pdf_files:
    path = os.path.join(raw_dir, filename)
    loader = PyPDFLoader(path)
    pages = loader.load()
    all_pages.extend(pages)
    print(f"Loaded {filename}: {len(pages)} pages")

print(f"\nTotal pages: {len(all_pages)}")

# fix broken and incorrect metadata titles from Word template placeholders
metadata_fixes = {
    "TS_38104.pdf": "3GPP TS 38.104",
    "TS_38212.pdf": "3GPP TS 38.212",
    "TS_38215.pdf": "3GPP TS 38.215",
    "TS_38843.pdf": "3GPP TR 38.843",
}

fixed_count = 0
for page in all_pages:
    filename = os.path.basename(page.metadata.get("source", ""))
    if filename in metadata_fixes:
        page.metadata["title"] = metadata_fixes[filename]
        fixed_count += 1

print(f"Metadata fixes applied to {fixed_count} pages")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)
chunks = splitter.split_documents(all_pages)
print(f"Total chunks: {len(chunks)}")

# embed in batches of 500 with retry on failure
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
client = chromadb.PersistentClient(path=chroma_path)
collection = client.get_or_create_collection("langchain")

batch_size = 500
total_batches = (len(chunks) + batch_size - 1) // batch_size

print(f"\nEmbedding {len(chunks)} chunks in {total_batches} batches of {batch_size}...")

for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]
    batch_num = i // batch_size + 1

    for attempt in range(3):
        try:
            texts = [doc.page_content for doc in batch]
            metadatas = [doc.metadata for doc in batch]
            ids = [str(i + j) for j in range(len(batch))]
            vectors = embeddings.embed_documents(texts)

            collection.add(
                embeddings=vectors,
                documents=texts,
                metadatas=metadatas,
                ids=ids
            )
            print(f"Batch {batch_num}/{total_batches} done ({len(batch)} chunks)")
            break

        except Exception as e:
            print(f"Batch {batch_num} attempt {attempt+1} failed: {e}")
            if attempt < 2:
                print("Retrying in 10 seconds...")
                time.sleep(10)
            else:
                print(f"Batch {batch_num} failed after 3 attempts")

print(f"\nDone. Total chunks in ChromaDB: {collection.count()}")

Loaded TS_38104.pdf: 454 pages
Loaded TS_38211.pdf: 171 pages
Loaded TS_38212.pdf: 327 pages
Loaded TS_38213.pdf: 331 pages
Loaded TS_38214.pdf: 356 pages
Loaded TS_38215.pdf: 38 pages
Loaded TS_38300.pdf: 314 pages
Loaded TS_38331.pdf: 1902 pages
Loaded TS_38401.pdf: 184 pages
Loaded TS_38410.pdf: 20 pages
Loaded TS_38843.pdf: 303 pages
Loaded TS_38873.pdf: 19 pages
Loaded TS_38912.pdf: 74 pages

Total pages: 4493
Metadata fixes applied to 1122 pages
Total chunks: 18187

Embedding 18187 chunks in 37 batches of 500...
Batch 1/37 done (500 chunks)
Batch 2/37 done (500 chunks)
Batch 3/37 done (500 chunks)
Batch 4/37 done (500 chunks)
Batch 5/37 done (500 chunks)
Batch 6/37 done (500 chunks)
Batch 7/37 done (500 chunks)
Batch 8/37 done (500 chunks)
Batch 9/37 done (500 chunks)
Batch 10/37 done (500 chunks)
Batch 11/37 done (500 chunks)
Batch 12/37 done (500 chunks)
Batch 13/37 done (500 chunks)
Batch 14/37 done (500 chunks)
Batch 15/37 done (500 chunks)
Batch 16/37 done (500 chunks)
Batch

In [2]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma(
    persist_directory=r"C:\Projects\3gpp-rag\chroma_db",
    embedding_function=embeddings
)

test_queries = [
    "What AI and ML techniques are proposed for the NR air interface?",
    "What are the MIMO enhancement techniques studied for NR?",
    "What are the base station conducted output power requirements?"
]

for query in test_queries:
    docs = vectorstore.similarity_search(query, k=2)
    print(f"Query: {query}")
    for doc in docs:
        print(f"  → {doc.metadata.get('title')} | Page {doc.metadata.get('page')}")
    print()

Query: What AI and ML techniques are proposed for the NR air interface?
  → 3GPP TS 38.401 | Page 35
  → 3GPP TS 38.300 | Page 250

Query: What are the MIMO enhancement techniques studied for NR?
  → 3GPP TR 38.912 | Page 22
  → 3GPP TR 38.912 | Page 2

Query: What are the base station conducted output power requirements?
  → 3GPP TS 38.104 | Page 2
  → 3GPP TS 38.104 | Page 12

